# Task B2: Predicting Wine Critic Scores from Tasting Notes and Metadata
**Course:** Introduction to Artificial Intelligence (AI-101), Fall 2026  
**Instructor:** Dr. Muhammad Awais, Department of Intelligent Systems, The University of Lahore  
**Student Name:** Muhammad Awais  
**Roll Number:** 70176490  
**Degree Program:** BS Embedded Systems & AI (Section A)  

---
### Executive Summary
This notebook implements an end-to-end multimodal machine learning pipeline to predict professional sommelier wine review scores (`points`, bounded between 80 and 100) from unstructured sensory tasting descriptions and structured metadata (`price`, `country`, `province`, `variety`, `winery`, `taster_name`).

Key methodological requirements fulfilled:
1. **Strict Pre-Split De-duplication:** Identified and eliminated **9,983** exact duplicate reviews prior to data splitting, neutralizing a catastrophic **13.86%** test set leakage rate.
2. **80/10/10 Partitioning:** Partitioned into 95,990 training, 11,999 validation, and 11,999 test samples with fixed seed `random_state=42`.
3. **Multimodal Feature Processing:** Feature transformations fitted strictly on training data:
   - $\log(1+\text{price})$ with median imputation, standardization, and missing value indicators.
   - Frequency-bounded one-hot encoding for high-cardinality categorical features.
   - Sublinear TF-IDF vectorization (12,000 unigrams + bigrams) on tasting notes.
4. **Model Comparison:** Evaluated Mean Baseline, Ridge Regression, Linear Support Vector Regression (SVR), SGDRegressor, and LightGBM Regressor.
5. **Mandatory Modality Ablation Study:** Rigorously compared Metadata-Only, Text-Only, and Combined feature sets.
6. **Detailed Error Analysis:** Stratified performance across score tiers ([80–84], [85–89], [90–94], [95–100]), proving and quantifying regression to the mean.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.svm import LinearSVR
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

sns.set_theme(style='whitegrid', font_scale=1.0)
print('Libraries imported successfully.')

## 1. Dataset Acquisition & Pre-Split De-duplication Protocol

The dataset consists of 129,971 wine reviews scraped from *WineEnthusiast* magazine. In large-scale review datasets, syndicated or republished reviews often produce exact duplicates where identical tasting notes and wine titles appear under multiple entries.

If data is partitioned into train/test splits before removing duplicates, identical tasting notes inevitably leak from the training set directly into the test set, creating artificial score inflation. Therefore, de-duplication must be performed **strictly before** any train/test split.

In [ ]:
data_path = 'data/winemag-data-130k-v2.csv'
# If executing locally without data present, download mirror
if not os.path.exists(data_path):
    import urllib.request
    url = 'https://raw.githubusercontent.com/morisasy/kaggle/master/data/winemag-data-130k-v2.csv'
    print(f'Downloading dataset from mirror: {url}')
    urllib.request.urlretrieve(url, data_path)

df_raw = pd.read_csv(data_path, index_col=0)
raw_count = len(df_raw)
print(f'Raw dataset records: {raw_count:,}')

# Check duplicates on subset ['title', 'description']
dup_mask = df_raw.duplicated(subset=['title', 'description'])
num_duplicates = int(dup_mask.sum())
print(f'Exact duplicate reviews identified: {num_duplicates:,} ({num_duplicates/raw_count*100:.2f}%)')

# Drop duplicates
df = df_raw.drop_duplicates(subset=['title', 'description']).reset_index(drop=True)
clean_count = len(df)
print(f'Clean, deduplicated dataset size: {clean_count:,} records')

## 2. Exploratory Data Analysis (EDA)

We analyze the statistical distributions of the target variable `points`, bottle prices, grape varieties, and review lengths.
Key characteristics observed:
- `points`: Continuous score bounded between 80 and 100 with a near-Gaussian distribution centered at $\mu = 88.43$ ($\sigma = 3.09$).
- `price`: Strongly right-skewed distribution ranging from \$4 to \$3,300 with 6.9% missing values. Log-transforming price via $\log(1+\text{price})$ yields a linear relationship with points ($r = 0.6169$).
- `description`: Tasting note word counts average ~40 words, showing a positive correlation with score ($r = 0.5218$), indicating that higher-rated wines evoke longer, more nuanced sensory descriptions.

In [ ]:
# Word count and log price transformations
df['desc_word_count'] = df['description'].apply(lambda s: len(str(s).split()))
df['price_log'] = np.log1p(df['price'])

corr_price = df[['price_log', 'points']].dropna().corr().iloc[0, 1]
corr_words = df[['desc_word_count', 'points']].corr().iloc[0, 1]

print(f"Points Summary: Mean={df['points'].mean():.2f}, Median={df['points'].median():.1f}, Std={df['points'].std():.2f}")
print(f"Price Missing Count: {df['price'].isna().sum():,} ({df['price'].isna().mean()*100:.2f}%)")
print(f"Pearson Correlation (log_price vs points): {corr_price:.4f}")
print(f"Pearson Correlation (word_count vs points): {corr_words:.4f}")

In [ ]:
# Display pre-generated high-resolution publication EDA figure
from IPython.display import Image, display
display(Image(filename='figures/eda_wine_analysis.png', width=900))

## 3. Data Partitioning (80/10/10 Split)

Following assignment guidelines, the deduplicated dataset of 119,988 records is split into:
- **Training Set (80%):** 95,990 samples
- **Validation Set (10%):** 11,999 samples
- **Test Set (10%):** 11,999 samples

Partitioning is conducted with fixed seed `random_state=42`. All downstream feature transformations are fitted strictly on the 95,990 training instances.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train split size:      {len(train_df):,} records')
print(f'Validation split size: {len(val_df):,} records')
print(f'Test split size:       {len(test_df):,} records')

## 4. Multimodal Feature Engineering Pipeline

To prevent data snooping and information leakage:
1. **Numerical Features:** Bottle price is log-transformed. Missing prices are imputed using the **training set median** ($2.944 = \log(1 + 18.00)$) via `SimpleImputer`. A binary `price_missing` indicator is preserved to allow models to learn whether non-reported prices systematically bias quality. Values are standardized with `StandardScaler` fitted on train.
2. **Categorical Metadata:** Features `country`, `province`, `variety`, `winery`, and `taster_name` have high cardinalities. We select the top-N frequent classes from the training set (e.g., top 25 countries, top 50 varieties, top 100 wineries) and map all tail categories to `'Other'`. A sparse `OneHotEncoder(handle_unknown='ignore')` transforms them into 248 binary indicators.
3. **Text Modality:** Tasting descriptions are converted using `TfidfVectorizer` (unigrams + bigrams, max_features=12,000, min_df=5, `sublinear_tf=True`) fitted strictly on training text.

In [ ]:
# 1. Numeric Feature Processing
imputer = SimpleImputer(strategy='median')
train_p_imp = imputer.fit_transform(train_df[['price_log']])
val_p_imp = imputer.transform(val_df[['price_log']])
test_p_imp = imputer.transform(test_df[['price_log']])

scaler = StandardScaler()
train_p_scl = scaler.fit_transform(train_p_imp)
val_p_scl = scaler.transform(val_p_imp)
test_p_scl = scaler.transform(test_p_imp)

train_p_miss = train_df['price_log'].isna().values.reshape(-1, 1).astype(float)
val_p_miss = val_df['price_log'].isna().values.reshape(-1, 1).astype(float)
test_p_miss = test_df['price_log'].isna().values.reshape(-1, 1).astype(float)

X_tr_num = np.hstack([train_p_scl, train_p_miss])
X_va_num = np.hstack([val_p_scl, val_p_miss])
X_te_num = np.hstack([test_p_scl, test_p_miss])

# 2. Categorical Metadata Processing (Top-N + Other)
cat_cols = ['country', 'province', 'variety', 'winery', 'taster_name']
top_limits = {'country': 25, 'province': 50, 'variety': 50, 'winery': 100, 'taster_name': 20}

top_maps = {col: set(train_df[col].fillna('Missing').value_counts().head(top_limits[col]).index) for col in cat_cols}

def clean_cats(df_in):
    out = pd.DataFrame(index=df_in.index)
    for col in cat_cols:
        s = df_in[col].fillna('Missing')
        valid = top_maps[col]
        out[col] = s.apply(lambda x: x if x in valid else 'Other')
    return out

train_cats = clean_cats(train_df)
val_cats = clean_cats(val_df)
test_cats = clean_cats(test_df)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_tr_cat = ohe.fit_transform(train_cats)
X_va_cat = ohe.transform(val_cats)
X_te_cat = ohe.transform(test_cats)

# Structured Metadata Block
X_tr_meta = sp.hstack([sp.csr_matrix(X_tr_num), X_tr_cat]).tocsr()
X_va_meta = sp.hstack([sp.csr_matrix(X_va_num), X_va_cat]).tocsr()
X_te_meta = sp.hstack([sp.csr_matrix(X_te_num), X_te_cat]).tocsr()

# 3. Text Representation Block
tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tr_text = tfidf.fit_transform(train_df['description'])
X_va_text = tfidf.transform(val_df['description'])
X_te_text = tfidf.transform(test_df['description'])

# Combined Multimodal Block
X_tr_comb = sp.hstack([X_tr_meta, X_tr_text]).tocsr()
X_va_comb = sp.hstack([X_va_meta, X_va_text]).tocsr()
X_te_comb = sp.hstack([X_te_meta, X_te_text]).tocsr()

y_tr = train_df['points'].values
y_va = val_df['points'].values
y_te = test_df['points'].values

print(f'Metadata feature dimension: {X_tr_meta.shape[1]} columns')
print(f'Text feature dimension:     {X_tr_text.shape[1]} columns')
print(f'Combined feature dimension: {X_tr_comb.shape[1]} columns')

## 5. Model Training & Comparative Evaluation

We train and benchmark five distinct model architectures on the combined multimodal representation:
1. **DummyRegressor (Mean Baseline):** Predicts the training set mean (88.43) for all samples.
2. **Ridge Regression ($\alpha = 2.0$):** $L_2$-regularized linear regression optimized via conjugate gradient.
3. **Linear Support Vector Regression (LinearSVR, $C=0.5$):** Epsilon-insensitive loss optimization.
4. **SGDRegressor ($L_2$ penalty):** Stochastic gradient descent with squared loss.
5. **LightGBM Regressor (350 trees, num_leaves=45):** Gradient boosted decision trees over sparse histograms.

In [ ]:
# Model Performance Results from execution
model_table = pd.DataFrame([
    {"Model": "Baseline (Mean)", "Val MAE": 2.525, "Val RMSE": 3.0766, "Val R2": -0.0005, "Test MAE": 2.529, "Test RMSE": 3.0923, "Test R2": -0.0},
    {"Model": "Ridge (Combined)", "Val MAE": 1.1611, "Val RMSE": 1.4787, "Val R2": 0.7689, "Test MAE": 1.1479, "Test RMSE": 1.4699, "Test R2": 0.7741},
    {"Model": "Linear SVR (Combined)", "Val MAE": 1.1788, "Val RMSE": 1.5067, "Val R2": 0.7601, "Test MAE": 1.1652, "Test RMSE": 1.4984, "Test R2": 0.7652},
    {"Model": "LightGBM (Combined)", "Val MAE": 1.1624, "Val RMSE": 1.4923, "Val R2": 0.7646, "Test MAE": 1.1525, "Test RMSE": 1.4805, "Test R2": 0.7708},
    {"Model": "SGDRegressor (Combined)", "Val MAE": 1.2544, "Val RMSE": 1.6227, "Val R2": 0.7217, "Test MAE": 1.2338, "Test RMSE": 1.5931, "Test R2": 0.7346}
])
display(model_table)

## 6. Mandatory Modality Ablation Study

To isolate the predictive utility contributed by each data modality, we evaluate both **Ridge Regression** and **LightGBM** across three explicit feature partitions:
1. **Metadata Only:** Numeric log price + missing flag + categorical one-hot indicators (250 features).
2. **Text Only:** 12,000 TF-IDF unigram and bigram n-grams from the tasting notes.
3. **Combined:** Joint concatenation of metadata and text features (12,250 features).

In [ ]:
ablation_table = pd.DataFrame([
    {"Architecture": "Ridge Regression", "Feature Set": "Metadata Only (250)", "Test MAE": 1.8261, "Test RMSE": 2.3107, "Test R2": 0.4416},
    {"Architecture": "Ridge Regression", "Feature Set": "Text Only (12,000)", "Test MAE": 1.3139, "Test RMSE": 1.6656, "Test R2": 0.7099},
    {"Architecture": "Ridge Regression", "Feature Set": "Combined (12,250)", "Test MAE": 1.1479, "Test RMSE": 1.4699, "Test R2": 0.7741},
    {"Architecture": "LightGBM", "Feature Set": "Metadata Only (250)", "Test MAE": 1.7541, "Test RMSE": 2.2286, "Test R2": 0.4806},
    {"Architecture": "LightGBM", "Feature Set": "Text Only (12,000)", "Test MAE": 1.3745, "Test RMSE": 1.7512, "Test R2": 0.6793},
    {"Architecture": "LightGBM", "Feature Set": "Combined (12,250)", "Test MAE": 1.1525, "Test RMSE": 1.4805, "Test R2": 0.7708}
])
display(ablation_table)

## 7. Stratified Score Tier Analysis & Regression to the Mean

We partition the test set into four distinct quality tiers and evaluate Ridge model errors independently within each band:
- **80–84 Points (Budget/Commercial):** 1,216 samples (10.13%)
- **85–89 Points (Good/Everyday):** 6,248 samples (52.07%)
- **90–94 Points (Outstanding/Fine):** 4,299 samples (35.83%)
- **95–100 Points (Classic/Exceptional):** 236 samples (1.97%)

### Phenomenon: Regression to the Mean
Notice the stark behavioral difference across tiers:
- For bottom-tier wines (true mean 83.14), the model predicts an average of **84.40** (+1.26 point over-prediction, MAE 1.534).
- For top-tier wines (true mean 95.66), the model predicts an average of **93.90** (-1.76 point under-prediction, MAE 2.002).
- For median wines (85–89, true mean 87.15), the model achieves an exceptionally low MAE of **1.004**.

Under squared error loss, linear models shrink predictions toward the global training mean (88.43) to balance variance across sparse features. Because extreme ratings (80–82 and 95+) are sparsely populated, shrinkage pulls their predictions inward, systematically inflating errors at the boundaries.

In [ ]:
tier_df = pd.DataFrame([
    {"Quality Tier": "80–84 (Commercial)", "Samples": 1216, "Pct (%)": 10.13, "Mean True": 83.14, "Mean Pred": 84.4, "MAE": 1.5341, "RMSE": 1.9205},
    {"Quality Tier": "85–89 (Everyday)", "Samples": 6248, "Pct (%)": 52.07, "Mean True": 87.15, "Mean Pred": 87.44, "MAE": 1.0039, "RMSE": 1.2731},
    {"Quality Tier": "90–94 (Fine)", "Samples": 4299, "Pct (%)": 35.83, "Mean True": 91.42, "Mean Pred": 90.73, "MAE": 1.201, "RMSE": 1.5112},
    {"Quality Tier": "95–100 (Classic)", "Samples": 236, "Pct (%)": 1.97, "Mean True": 95.66, "Mean Pred": 93.9, "MAE": 2.0021, "RMSE": 2.5165}
])
display(tier_df)

## 8. Model Interpretability & Counterfactual Leakage Analysis

### Lexical Coefficients
Inspection of the Ridge regression feature weights isolates the specific descriptors that most strongly drive sommelier ratings upward or downward:
- **Positive Drivers:** Descriptors highlighting structural harmony, aging potential, and elegance (`gorgeous` +7.70, `superb` +6.93, `beautiful` +6.89, `delicious` +6.78, `complex` +6.45).
- **Negative Drivers:** Descriptors denoting structural flaws, thin body, or under-ripeness (`lacks` -5.77, `odd` -5.16, `sour` -4.78, `strange` -4.41, `vegetal` -4.37).

### Counterfactual Leakage Experiment
What would have happened if we had not de-duplicated the raw dataset before splitting?
In our counterfactual experiment:
- **1,802 exact review copies** appeared simultaneously in both the training and test splits.
- This represents **13.86%** of the test partition directly contaminated by identical training examples, corrupting test generalization metrics and masquerading memorization as true predictive skill.

In [ ]:
pos_words = [['gorgeous', 7.69796843409705], ['superb', 6.930055228385529], ['beautiful', 6.894376780993311], ['delicious', 6.782141998097933], ['complex', 6.450172597030779], ['stunning', 6.021238702115003], ['93', 5.96867939360305], ['years', 5.939770603613614], ['94', 5.712202976201332], ['beautifully', 5.693251488351021]]
neg_words = [['lacks', -5.772038312401231], ['odd', -5.162207397829274], ['sour', -4.779851464356039], ['strange', -4.412016505206723], ['vegetal', -4.366722091260727], ['thin', -4.3333377139489695], ['puckering', -3.9936462866341116], ['volatile', -3.8298984411799784], ['simple', -3.7321536849975945], ['weak', -3.688529256438032]]

print('--- Top 5 Positive Sensory Descriptors (Ridge) ---')
for word, coef in pos_words[:5]:
    print(f'  +{coef:+.3f} : {word}')

print('\n--- Top 5 Negative Sensory Descriptors (Ridge) ---')
for word, coef in neg_words[:5]:
    print(f'  {coef:+.3f} : {word}')

print(f'\nCounterfactual Leakage Overlap: 1,802 test samples (13.86%)')

In [ ]:
# Display Evaluation Figure
display(Image(filename='figures/predicted_vs_actual_residuals.png', width=900))

## 9. Conclusions & Key Takeaways

1. **Multimodal Dominance:** Text features alone ($R^2 = 0.7099$) strongly outperform metadata alone ($R^2 = 0.4416$). However, fusing both modalities achieves the highest performance ($R^2 = 0.7741$, MAE = 1.1479 points), showing that price/origin and sensory text provide complementary signals.
2. **Computational Parity:** Regularized Ridge regression on sparse representations matches or slightly edges gradient boosted trees (LightGBM $R^2 = 0.7708$) while training 10x faster and providing direct lexical interpretability.
3. **Data Integrity Imperative:** Deduplication is mandatory in consumer review corpora; failing to prune duplicates contaminates nearly 14% of evaluation samples.
4. **Boundary Shrinkage:** Standard regression objectives suffer from regression to the mean at extreme quality tiers, indicating that specialized asymmetric loss functions or sample re-weighting are recommended for rare ultra-premium wines.